# 01 — Build the evaluation-driver catalog

This notebook processes every Markdown case in `data/train` in filename order.
For each case, the OpenAI model compares the case with the current catalog,
reports matching drivers and gaps, and proposes additions. Valid additions are
automatically applied and the catalog is checkpointed after every case.

The catalog is universal across ML/AI project types and includes technical and
organizational implementation work. It does not estimate cost or aggregate
complexity. Images and linked resources are intentionally ignored in this MVP.

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Literal

import yaml
from openai import OpenAI
from pydantic import BaseModel, Field, model_validator


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config" / "pipeline.yaml").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repo_root()
CONFIG = yaml.safe_load((ROOT / "config" / "pipeline.yaml").read_text())
MODEL = os.getenv("OPENAI_MODEL", CONFIG["openai"]["model"])
client = OpenAI()


def read_case(path: Path) -> str:
    # Images are intentionally outside the MVP scope. Ignore extracted-image appendices.
    text = path.read_text(encoding="utf-8")
    return text.split("## Extracted images", 1)[0].strip()


def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def append_jsonl(path: Path, value: dict) -> None:
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(value, ensure_ascii=False) + "\n")


def utc_run_id(prefix: str) -> str:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    return f"{prefix}_{stamp}"


In [ ]:
class Category(BaseModel):
    id: str
    label: str
    description: str
    work_meaning: str
    min_inclusive: float | None = None
    min_exclusive: float | None = None
    max_inclusive: float | None = None
    max_exclusive: float | None = None


class Driver(BaseModel):
    id: str
    name: str
    area: str
    description: str
    rationale: str
    source_type: Literal["numeric", "categorical", "binary"]
    applies_when: str
    does_not_apply_when: str
    categories: list[Category] = Field(min_length=2)
    tags: list[str]
    introduced_by_cases: list[str]

    @model_validator(mode="after")
    def validate_binary_driver(self):
        if self.source_type == "binary" and len(self.categories) != 2:
            raise ValueError("A binary driver must have exactly two categories.")
        if len({category.id for category in self.categories}) != len(self.categories):
            raise ValueError("Category IDs must be unique within a driver.")
        if self.source_type == "numeric":
            intervals = []
            for category in self.categories:
                lower = category.min_inclusive if category.min_inclusive is not None else category.min_exclusive
                upper = category.max_inclusive if category.max_inclusive is not None else category.max_exclusive
                if lower is None and upper is None:
                    raise ValueError("Numeric categories must define at least one boundary.")
                intervals.append((float("-inf") if lower is None else lower, float("inf") if upper is None else upper, category))
            intervals.sort(key=lambda item: item[0])
            for (_, previous_upper, previous), (current_lower, _, current) in zip(intervals, intervals[1:]):
                touching_inclusively = (
                    previous_upper == current_lower
                    and previous.max_inclusive is not None
                    and current.min_inclusive is not None
                )
                if previous_upper > current_lower or touching_inclusively:
                    raise ValueError(f"Numeric categories overlap: {previous.id}, {current.id}")
        return self


class ExistingDriverMatch(BaseModel):
    driver_id: str
    evidence: list[str]
    explanation: str


class MissingCategory(BaseModel):
    driver_id: str
    category: Category
    evidence: list[str]
    reason: str


class ProposedDriver(BaseModel):
    driver: Driver
    evidence: list[str]
    overlap_check: list[str]


class BuildAnalysis(BaseModel):
    case_id: str
    project_summary: str
    scope_comments: list[str]
    relevant_existing_drivers: list[ExistingDriverMatch]
    insufficient_information_driver_ids: list[str]
    missing_categories: list[MissingCategory]
    missing_drivers: list[ProposedDriver]
    notes: list[str]


In [ ]:
CATALOG_PATH = ROOT / CONFIG["paths"]["catalog"]
PROMPT_DIR = ROOT / CONFIG["paths"]["prompts"]
SYSTEM_PROMPT = (PROMPT_DIR / "build_system.md").read_text(encoding="utf-8")
USER_PROMPT = (PROMPT_DIR / "build_user.md").read_text(encoding="utf-8")


def load_catalog() -> dict:
    return yaml.safe_load(CATALOG_PATH.read_text(encoding="utf-8"))


def save_catalog(catalog: dict) -> None:
    CATALOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CATALOG_PATH.write_text(
        yaml.safe_dump(catalog, sort_keys=False, allow_unicode=True),
        encoding="utf-8",
    )


def analyze_case(case_path: Path, catalog: dict) -> BuildAnalysis:
    case_id = case_path.stem.split("_disdoc", 1)[0]
    case_text = read_case(case_path)
    user_prompt = USER_PROMPT.format(
        catalog=yaml.safe_dump(catalog, sort_keys=False, allow_unicode=True),
        case_id=case_id,
        case_path=case_path.relative_to(ROOT),
        case_text=case_text,
    )
    response = client.responses.parse(
        model=MODEL,
        input=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        text_format=BuildAnalysis,
    )
    if response.output_parsed is None:
        raise RuntimeError(f"The model did not return a parsed result for {case_id}.")
    return response.output_parsed


def apply_analysis(catalog: dict, analysis: BuildAnalysis) -> list[dict]:
    changes: list[dict] = []
    by_id = {driver["id"]: driver for driver in catalog["drivers"]}

    for proposal in analysis.missing_categories:
        driver = by_id.get(proposal.driver_id)
        if driver is None:
            raise ValueError(f"Unknown driver in category proposal: {proposal.driver_id}")
        known_categories = {category["id"] for category in driver["categories"]}
        if proposal.category.id not in known_categories:
            candidate_categories = [*driver["categories"], proposal.category.model_dump(exclude_none=True)]
            Driver.model_validate({**driver, "categories": candidate_categories})
            driver["categories"] = candidate_categories
            changes.append({
                "change_type": "ADD_CATEGORY",
                "driver_id": proposal.driver_id,
                "category_id": proposal.category.id,
                "reason": proposal.reason,
            })

    for proposal in analysis.missing_drivers:
        driver = proposal.driver
        if driver.id not in by_id:
            payload = driver.model_dump(exclude_none=True)
            if analysis.case_id not in payload["introduced_by_cases"]:
                payload["introduced_by_cases"].append(analysis.case_id)
            catalog["drivers"].append(payload)
            by_id[driver.id] = payload
            changes.append({
                "change_type": "ADD_DRIVER",
                "driver_id": driver.id,
                "overlap_check": proposal.overlap_check,
            })

    if changes:
        catalog["catalog_version"] += 1
    return changes


def render_case_build_report(analysis: BuildAnalysis, changes: list[dict]) -> str:
    lines = [
        f"## {analysis.case_id}", "", analysis.project_summary, "",
        "### Scope comments", "",
    ]
    lines.extend([f"- {item}" for item in analysis.scope_comments] or ["- None."])
    lines.extend(["", "### Relevant existing drivers", ""])
    if analysis.relevant_existing_drivers:
        for match in analysis.relevant_existing_drivers:
            lines.extend([
                f"- **{match.driver_id}** — {match.explanation}",
                *[f"  - Evidence: “{evidence}”" for evidence in match.evidence],
            ])
    else:
        lines.append("- None.")
    lines.extend(["", "### Applied additions", ""])
    lines.extend(
        [f"- `{change['change_type']}`: `{change['driver_id']}`" for change in changes]
        or ["- None."]
    )
    lines.extend(["", "### Insufficient information", ""])
    lines.extend(
        [f"- `{driver_id}`" for driver_id in analysis.insufficient_information_driver_ids]
        or ["- None."]
    )
    return "\n".join(lines) + "\n"


## Run the build

This cell makes paid OpenAI API calls. With approximately 80 cases, review the
configured model before running. A new timestamped log directory is created for
every run; the tracked catalog is updated in place.

In [ ]:
train_dir = ROOT / CONFIG["paths"]["train_cases"]
case_paths = sorted(train_dir.glob("*.md"))
if not case_paths:
    raise FileNotFoundError(f"No Markdown cases found in {train_dir}")

run_id = utc_run_id("build")
run_dir = ROOT / CONFIG["paths"]["build_artifacts"] / run_id
run_dir.mkdir(parents=True, exist_ok=False)
analysis_log = run_dir / "case_analysis.jsonl"
changes_log = run_dir / "catalog_changes.jsonl"
report_path = run_dir / "case_analysis.md"
report_path.write_text(f"# Catalog build report: {run_id}\n\n", encoding="utf-8")

catalog = load_catalog()
starting_version = catalog["catalog_version"]
for index, case_path in enumerate(case_paths, start=1):
    version_before = catalog["catalog_version"]
    analysis = analyze_case(case_path, catalog)
    if CONFIG["catalog"]["auto_apply_additions"]:
        changes = apply_analysis(catalog, analysis)
        save_catalog(catalog)  # Checkpoint after every case.
    else:
        changes = []

    log_record = {
        "run_id": run_id,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "case_path": str(case_path.relative_to(ROOT)),
        "case_sha256": sha256_text(read_case(case_path)),
        "model": MODEL,
        "catalog_version_before": version_before,
        "catalog_version_after": catalog["catalog_version"],
        "analysis": analysis.model_dump(),
        "applied_changes": changes,
    }
    append_jsonl(analysis_log, log_record)
    for change in changes:
        append_jsonl(changes_log, {"run_id": run_id, "case_id": analysis.case_id, **change})
    with report_path.open("a", encoding="utf-8") as report:
        report.write(render_case_build_report(analysis, changes) + "\n")
    print(f"[{index}/{len(case_paths)}] {analysis.case_id}: {len(changes)} change(s)")

summary = {
    "run_id": run_id,
    "model": MODEL,
    "case_count": len(case_paths),
    "catalog_version_before": starting_version,
    "catalog_version_after": catalog["catalog_version"],
    "driver_count": len(catalog["drivers"]),
}
(run_dir / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
summary
